# Introdução à Teoria dos Grafos — análise temporal integrada

**Atividade:** redes temporais, centralidades, comunidades e visualização moderna  
**Autor:** Diogo da Silva Rego  
**Matrícula:** 20240045381  
**Professora:** Ana Flávia  
**Instituição:** CCEN/UFPB — Estatística DCE

Este notebook integra três frentes: terremotos, o notebook `college_msg.ipynb` e as redes ponderadas das temporadas **2 a 8** de Game of Thrones. A temporada 1 não foi inventada porque não foi fornecida nos arquivos desta atividade.

A camada moderna usa `python-igraph` para análise, `leidenalg` para comunidades, NetworkX como referência e exportação GEXF/GraphML para exploração no Gephi. As posições dos grafos GOT são globais e fixas entre temporadas, e os HTMLs são complementos às figuras estáticas.


In [ ]:
# 1. Instalação reprodutível no Google Colab
!pip -q install \
  "python-igraph==1.0.0" \
  "leidenalg==0.12.0" \
  "networkx==3.7" \
  "plotly==6.9.0" \
  "seaborn==0.13.2" \
  "snakemake==9.27.0" \
  geopandas pyogrio

import sys
print(sys.version)
import igraph, leidenalg, networkx, plotly
print("igraph:", igraph.__version__)
print("leidenalg:", getattr(leidenalg, "__version__", "versão não exposta"))
print("networkx:", networkx.__version__)
print("plotly:", plotly.__version__)


In [ ]:
# 2. Upload dos dados e dos módulos Python
from pathlib import Path
from google.colab import files

arquivos = [
    "terremotos.csv",
    "college_msg.ipynb",
    "earthquake_got_temporal_analysis.py",
    "earthquakes_temporal_modern.py",
    "got_temporal_modern.py",
    "college_msg_temporal.py",
    *[f"got-s{i}-edges.csv" for i in range(2, 9)],
]
# O arquivo duplicado da temporada 2 é opcional, mas será auditado quando enviado.
opcionais = ["got-s2-edges(1).csv", "pasted_content_6.txt", "terremotos.ipynb"]
missing = [name for name in arquivos if not Path("/content", name).exists()]
if missing:
    print("Envie os arquivos obrigatórios:", ", ".join(missing))
    files.upload()
missing = [name for name in arquivos if not Path("/content", name).exists()]
if missing:
    raise FileNotFoundError(f"Arquivos ainda ausentes: {missing}")
print("Entradas e módulos disponíveis.")


In [ ]:
# 3. Análise moderna das temporadas GOT S2–S8
from pathlib import Path
import json
from got_temporal_modern import analyze_got

BASE = Path("/content")
RESULTADOS = Path("/content/resultados_atividade_integrada")
GOT_OUT = RESULTADOS / "got_s02_s08"
GOT_OUT.mkdir(parents=True, exist_ok=True)
duplicado = BASE / "got-s2-edges(1).csv"
resumo_got = analyze_got(BASE, GOT_OUT, duplicado if duplicado.exists() else None)
print(json.dumps(resumo_got, ensure_ascii=False, indent=2))


In [ ]:
# 4. Análise de terremotos — intervalo de cinco dias mais ativo
from earthquakes_temporal_modern import run_earthquakes

EQ_OUT = RESULTADOS / "terremotos"
resumo_eq = run_earthquakes(BASE / "terremotos.csv", EQ_OUT, cell_size_degrees=10.0)
print(json.dumps(resumo_eq, ensure_ascii=False, indent=2))


In [ ]:
# 5. Análise temporal do notebook college_msg.ipynb
from college_msg_temporal import run

COLLEGE_OUT = RESULTADOS / "college_msg"
resumo_college = run(COLLEGE_OUT, window_days=7)
print(json.dumps(resumo_college, ensure_ascii=False, indent=2))


In [ ]:
# 6. Tabelas e visualizações principais
from IPython.display import display, Image, HTML
import pandas as pd

print("
Top personagens por centralidade combinada média:")
display(pd.read_csv(GOT_OUT / "tabelas/centralidades_por_personagem.csv").groupby("character", as_index=False)["centrality_combined"].mean().sort_values("centrality_combined", ascending=False).head(15))
print("
Métricas das temporadas S2–S8:")
display(pd.read_csv(GOT_OUT / "tabelas/metricas_por_temporada.csv"))
print("
Primeiras janelas do CollegeMsg:")
display(pd.read_csv(COLLEGE_OUT / "tabelas/college_msg_metricas_por_janela.csv").head())

display(Image(filename=str(GOT_OUT / "figuras/evolucao_temporal_metricas.png")))
display(Image(filename=str(GOT_OUT / "figuras/heatmap_centralidades_top15.png")))
display(Image(filename=str(EQ_OUT / "mapa_mundi_intervalo_cinco_dias.png")))
display(Image(filename=str(COLLEGE_OUT / "figuras/college_msg_evolucao_rede.png")))
print("HTML GOT:", GOT_OUT / "interativo/got_temporal_s02_s08_interativo.html")


## Leitura metodológica

`Weight` representa força/frequência da interação. Para caminhos ponderados, o código cria separadamente `cost = 1 / weight_strength`; assim, força de conexão e custo de deslocamento não são confundidos. As arestas são não direcionadas porque os arquivos não fornecem emissor e receptor semanticamente distintos.

A análise de comunidades usa Leiden com objetivo RBConfiguration, três seeds e três resoluções. O resultado principal é acompanhado de uma análise de sensibilidade. A posição dos personagens é calculada uma vez no grafo agregado S2–S8 e reutilizada nos snapshots.

O notebook recebido como `college_msg.ipynb` aponta para o dataset `email-Eu-core-temporal.txt.gz` do Stanford SNAP. Essa origem é registrada na saída e no README para evitar confundir o nome da atividade com a fonte efetivamente carregada.

> Autor: Diogo Rego. As figuras estáticas e interativas incluem legendas explicativas e crédito autoral.
